# Application of Radial Equilibrium Equation for a Rotor - Direct Problem
## Adaptation of the code RE-ANAL by Lewis (Turbomachines Performance Analysis)

This notebook implements the **direct analysis (RE-ANAL) problem** for radial equilibrium in axial turbomachinery, as described in **Lewis Chapter 5 (Section 5.3.2)**. 
 
Unlike the *inverse (design)* problem (`RE-DES`), where we specify a desired swirl velocity $c_\theta(r)$ and calculate the required blade geometry, the *direct (analysis)* problem starts with a **fixed blade geometry** (defined by the relative outlet flow angle $\beta_2(r)$) and predicts the resulting axial velocity profile $c_x(r)$, absolute tangential velocity $c_\theta(r)$, and overall fan performance.

The necessary modules are imported

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.precision', 3)
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid, trapezoid

Let's input the general dimensions and target parameters of the fan. These are consistent with the **TXBR-250 ECOWATT** prototype fan studied in previous sessions.

In [ ]:
rho = 1.2                           # Density of air, kg/m³
Dh = 0.064                          # Diameter of hub, m
rh = Dh/2                           # Radius of hub, m
Dt = 0.25                           # Diameter of tip, m
rt = Dt/2                           # Radius of tip, m
h = rh/rt                           # Hub-tip ratio
rrms = np.sqrt(0.5*(rh*rh+rt*rt))   # RMS radius, m
n = 10                              # number of outputs
m = 600                             # number of interpolation points
r = np.linspace(rh,rt,m)            # Discretization of the radius for interpolation, m
Qdata = 1716                        # Flow rate, m³/h
Qdata = Qdata/3600                  # Flow rate, m³/s
omega = 2275                        # Rotational speed, rpm
omega = omega*np.pi/30              # Rotational speed, rad/s
Delta_p0_target = 75                # Pressure rise, Pa

ctrms = Delta_p0_target/(rho*omega*rrms)         # c_theta,rms, m/s
cxm = Qdata/(np.pi*(rt*rt-rh*rh))   # c_x,rms, m/s
print("Flow rate = {:0.4f} m³/s".format(Qdata))
print("Hub to tip ratio = {:0.4f}".format(h))
print("rms = {:.4f} m".format(rrms))
print("ctheta_rms = {:.4f} m/s".format(ctrms))
print("cx_rms = {:.4f} m/s".format(cxm))


Flow rate = 0.5000 m³/s
Hub to tip ratio = 0.2560
rms = 0.0912 m
ctheta_rms = 2.8754 m/s
cx_rms = 10.9003 m/s


In a direct analysis problem, the blade geometry is defined by the outlet flow angle $\beta_2(r)$ at several radial stations.
 
Below, we define the relative flow angles $\beta_2$ at 10 radial stations. 

In [44]:
rdata = np.linspace(rh, rt, 10)

# Relative outlet flow angle beta2 (degrees) from a successful design run
# Note: For a stator analysis, setting omega = 0 and inputting absolute flow angles alpha2 works identically!
# The following beta2_input values are from a successful design run, just copy-pasted here for analysis purposes.
beta2_input =np.array([
    31.46458518,
    49.72333857,
    57.8357223 , 
    61.13792921, 
    62.10363937,
    61.98550721, 
    61.40175539, 
    60.64550684, 
    59.85194521, 
    59.08138468
    ])
# Display input geometry
df_geom = pd.DataFrame({
    "Station Radius (m)": rdata,
    "beta_2 Flow Angle (deg)": beta2_input
})
df_geom

,Station Radius (m),beta_2 Flow Angle (deg)
0,0.032,31.465
1,0.042,49.723
2,0.053,57.836
3,0.063,61.138
4,0.073,62.104
5,0.084,61.986
6,0.094,61.402
7,0.104,60.646
8,0.115,59.852
9,0.125,59.081


From Lewis Section 5.3.2, the first-order linear differential equation governing the axial velocity downstream of a rotor $c_{x2}(r)$ is:

$$\frac{\text{d}c_{x2}}{\text{d}r} + f_1(r) c_{x2} = f_2(r)$$
 
Where:
$$f_1(r) = \frac{\tan \beta_2}{r (1 + \tan^2 \beta_2)} \frac{\text{d}(r \tan \beta_2)}{\text{d}r}$$
$$f_2(r) = \frac{2 \omega \tan \beta_2}{1 + \tan^2 \beta_2}$$
 
The solution is found iteratively by integrating:
$$c_{x2}(r) = L(r, c_{x2}) + K_1$$
 
Where $L(r, c_{x2}) = \int_{r_h}^r \left( -c_{x2} f_1(r) + f_2(r) \right) \text{d}r$, and the integration constant $K_1$ is updated to satisfy the target mass flow rate $Q$.


In [45]:
# Interpolate input beta2 distribution onto the fine integration grid
beta2_spline = CubicSpline(rdata, beta2_input)
beta2 = beta2_spline(r)
tanbeta2 = np.tan(np.radians(beta2))

# Pre-calculate terms for f1 and f2 evaluated at midpoints rm
rm = 0.5 * (r[:-1] + r[1:])
tanbeta2m = 0.5 * (tanbeta2[:-1] + tanbeta2[1:])

# Calculate the derivative d(r * tan(beta2)) / dr using central/forward differences on the fine grid
dr = r[1] - r[0]
r_tanbeta2 = r * tanbeta2
drtanbeta2 = (r_tanbeta2[1:] - r_tanbeta2[:-1]) / dr

# Evaluate f1 and f2 at midpoints rm
f1 = (tanbeta2m / rm) * drtanbeta2 / (1 + tanbeta2m**2)
f2 = 2 * omega * tanbeta2m / (1 + tanbeta2m**2)


We define the integration operator $L(r, c_x)$

In [46]:
def compute_L(cx_arr):
    # Midpoint axial velocity for integration step
    cx_m = 0.5 * (cx_arr[:-1] + cx_arr[1:])
    integrand = -cx_m * f1 + f2
    L_arr = np.zeros(m)
    L_arr[1:] = np.cumsum(integrand * dr)
    return L_arr

## The Iteration Loop

We solve the implicit relationship for $c_{x2}(r)$ and match the target flow rate $Q_{\text{data}}$ using the successive approximation scheme (Lewis Equation 5.47).

In [47]:
# Initialize axial velocity with the mean value
cx_anal = np.full(m, cxm)

# Initial integration of L
L = compute_L(cx_anal)
Lrms = CubicSpline(r, L)(rrms)
k1 = cxm - Lrms  # First approximation of K1

max_iter = 100
tolerance = 1e-6

print(f"{'Iteration':^12} | {'K1':^12} | {'K2':^12} | {'Error (%)':^12}")
print("—" * 55)

for j in range(1, max_iter + 1):
    cxnew = L + k1
    cx_anal = 0.5 * (cx_anal + cxnew) # Under-relaxation for stable convergence
    L = compute_L(cx_anal)
    
    # Calculate mass flow integral (r * L) over r
    integral_rL = trapezoid(r * L, r)
    k2 = cxm - 2 * integral_rL / (rt**2 - rh**2)
    
    # Check convergence error
    error = np.abs((k1 - k2) / k1)
    k1 = 0.5 * (k1 + k2)
    
    if j % 5 == 0 or error < tolerance:
        print(f"{j:^12d} | {k1:12.6f} | {k2:12.6f} | {error * 100:12.2e}")
        
    if error < tolerance:
        print(f"\nConvergence achieved successfully in {j} iterations!")
        break

 Iteration   |      K1      |      K2      |  Error (%)  
———————————————————————————————————————————————————————
     5       |    10.039424 |     9.709560 |     6.36e+00
     10      |     9.060084 |     8.930117 |     2.83e+00
     15      |     8.676394 |     8.625384 |     1.17e+00
     20      |     8.525548 |     8.505479 |     4.70e-01
     25      |     8.466191 |     8.458293 |     1.86e-01
     30      |     8.442833 |     8.439725 |     7.36e-02
     35      |     8.433642 |     8.432419 |     2.90e-02
     40      |     8.430025 |     8.429543 |     1.14e-02
     45      |     8.428601 |     8.428412 |     4.49e-03
     50      |     8.428041 |     8.427967 |     1.77e-03
     55      |     8.427821 |     8.427791 |     6.96e-04
     60      |     8.427734 |     8.427723 |     2.74e-04
     65      |     8.427700 |     8.427695 |     1.08e-04
     66      |     8.427696 |     8.427692 |     8.94e-05

Convergence achieved successfully in 66 iterations!


Now that the velocity field $c_x(r)$ has been determined, we can calculate the local absolute tangential velocity $c_\theta(r)$ and local total pressure rise $\Delta p_0(r)$ across the blade span:

$$c_{\theta}(r) = \omega r - c_x(r) \tan \beta_2(r)$$
$$\Delta p_0(r) = \rho \omega r c_\theta(r)$$

In [48]:
# Calculate local tangential velocity and pressure rise on the fine grid
ctheta_anal = omega * r - cx_anal * tanbeta2
Delta_p0 = rho * omega * r * ctheta_anal

# Integrate to find the overall calculated flow rate and area-averaged pressure rise
Q_calc = 2 * np.pi * trapezoid(r * cx_anal, r)
Delta_p0_avg = 2 * trapezoid(r * Delta_p0, r) / (rt**2 - rh**2)

print(f"\nOverall Fan Performance Summary:")
print(f"Calculated Flow Rate Q     = {Q_calc * 3600:.1f} m³/h (Target: {Qdata*3600:.1f} m³/h)")
print(f"Flow Rate Error            = {np.abs(Qdata - Q_calc)/Qdata * 100:.2e}%")
print(f"Area-weighted Avg Pressure = {Delta_p0_avg:.2f} Pa (Target: {Delta_p0_target:.1f} Pa)")
print(f"Pressure Rise Error        = {np.abs(Delta_p0_target - Delta_p0_avg)/Delta_p0_target * 100:.2f}%")


Overall Fan Performance Summary:
Calculated Flow Rate Q     = 1800.0 m³/h (Target: 1800.0 m³/h)
Flow Rate Error            = 5.24e-05%
Area-weighted Avg Pressure = 57.14 Pa (Target: 75.0 Pa)
Pressure Rise Error        = 23.81%


## Results Table (Pandas DataFrame)
Let's extract the radial solutions at the 10 output sections to display them in a clean table.

In [36]:
cx_sections = CubicSpline(r, cx_anal)(rdata)
ct_sections = CubicSpline(r, ctheta_anal)(rdata)
dp0_sections = CubicSpline(r, Delta_p0)(rdata)
beta1_sections = np.rad2deg(np.arctan(omega * rdata / cx_sections))

df_results = pd.DataFrame({
    "Radius (m)": rdata,
    "c_x (m/s)": cx_sections,
    "c_theta (m/s)": ct_sections,
    "beta_1 (deg)": beta1_sections,
    "beta_2 (deg)": beta2_input,
    "Local Delta p0 (Pa)": dp0_sections
})

df_results

,Radius (m),c_x (m/s),c_theta (m/s),beta_1 (deg),beta_2 (deg),Local Delta p0 (Pa)
0,0.032,1.256,6.855,80.642,31.465,62.710
1,0.042,2.826,6.750,74.348,49.723,81.698
2,0.053,4.001,6.184,72.313,57.836,93.117
3,0.063,5.099,5.757,71.235,61.138,103.690
4,0.073,6.310,5.551,70.141,62.104,116.376
5,0.084,7.659,5.536,68.980,61.986,132.423
6,0.094,9.118,5.670,67.847,61.402,152.372
7,0.104,10.652,5.916,66.802,60.646,176.472
8,0.115,12.237,6.249,65.870,59.852,204.845
9,0.125,13.855,6.647,65.050,59.081,237.549
